# Diagnose change in ocean-driven SST over time

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## funcs

In [ ]:
def get_nino_merged(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(190, 270))
    return x.sel(idx).mean(["longitude", "latitude"])



def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])

## Load data

In [ ]:
Th_3d = src.utils.load_cesm_indices_3d()

In [ ]:
## load indices
Th = src.utils.load_cesm_indices()

## load 3d indices
# Th_3d = src.utils.load_cesm_indices()



In [ ]:
forced, anom = src.utils.load_consolidated()

## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]
nhf_m = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_nino_merged)[
    "nhf"
]
nhf_e = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_Te)["nhf"]
T_m = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], fn=get_nino_merged)[
    "sst"
]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_34 = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)
Q_m = nhf_m * sec_per_yr / (rho * Cp * H)
Q_e = nhf_e * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q_34"] = Q_34.sel(time=Th.time)
Th["Q_3"] = Q_3.sel(time=Th.time)
Th["Q_e"] = Q_e.sel(time=Th.time)
Th["Q_m"] = Q_m.sel(time=Th.time)
Th["T_m"] = T_m.sel(time=Th.time)

#### window data

In [ ]:
## window
Th = src.utils.get_windowed(Th, stride=120, window_size=480)

## remove median
Th = Th.groupby("time.month") - Th.groupby("time.month").median(["time","member"])

## compute

In [ ]:
def get_spaghetti(idx, data, peak_month, event_type=None, q=0.75, is_warm=True):
    """
    Get hovmoller composite based on specified:
    - data: used to compute index/make composite
    - peak_month: month to center composite on
    - q: quantile threshold for composite
    """

    ## handle warm/cold case
    if is_warm:
        kwargs = dict(q=q, check_cutoff=lambda x, cut: x > cut)
    else:
        kwargs = dict(q=1 - q, check_cutoff=lambda x, cut: x < cut)

    ## kwargs for composite
    kwargs = dict(
        kwargs,
        avg=False,
        peak_month=peak_month,
        idx=idx,
        data=data,
        event_type=event_type,
    )

    ## composite of projected data
    spag = src.utils.make_composite(**kwargs)

    return spag

In [ ]:
## empty arrays to hold results
spags_warm = []
spags_cold = []


## loop thru years
for year in tqdm.tqdm(Th.year):

    ## get data
    data_y = Th.sel(year=year)

    ## shared args
    kwargs = dict(idx=data_y["T_34"], data=data_y, peak_month=1, q=.75)

    ## compute composites
    spags_warm.append(
        get_spaghetti(is_warm=True, **kwargs)
    )
    spags_cold.append(
        get_spaghetti(is_warm=False, **kwargs)
    )

## put in xarray
spags_warm = xr.concat(spags_warm, dim=Th.year)
spags_cold = xr.concat(spags_cold, dim=Th.year)

Compute GR

In [ ]:
fig,ax = plt.subplots(figsize=(4,4))
ax.scatter(
    spags_warm["T_3"].sel(lag=-6).isel(year=0),
    spags_warm["h"].sel(lag=-6).isel(year=0),
    s=10,
    # 2*spags_warm["h"].sel(lag=-6).isel(year=0)-spags_warm["h_w"].sel(lag=-6).isel(year=0),
)
ax.axvline(0, c="k")
ax.axhline(0, c="k")

In [ ]:
def get_dSST(spags, norm=False, t0=-6, t1=0, thresh=.25):
    """get normalized"""

    ## normalize
    if norm:

        ## get sign
        sign = np.sign(spags.sel(lag=0).mean("sample"))
    
        ## subset for same sign
        same_sign = (spags * sign) > thresh
        spags_ = spags.where(same_sign, other=np.nan)

        ## differentiate
        ddt_SST = spags_.differentiate("lag")

        ## normalize and integrate
        dSST = sign * (ddt_SST/spags_).sel(lag=slice(t0,t1)).sum("lag")

    else:
        ## get dSST
        dSST = spags.sel(lag=t1)-spags.sel(lag=t0)

    return dSST

In [ ]:
## get normal
kwargs = dict(norm = False, t0=-6, t1=0, thresh=0)
dSST_warm = get_dSST(spags_warm, **kwargs)
dSST_cold = get_dSST(spags_cold, **kwargs)


## norm kwargs
kwargs = dict(norm = True, t0=-6, t1=-1, thresh=1)
dSST_warm_norm = get_dSST(spags_warm, **kwargs)
dSST_cold_norm = get_dSST(spags_cold, **kwargs)

for [dSST_w, dSST_c] in [
    [dSST_warm, dSST_cold], [dSST_warm_norm, dSST_cold_norm],
]:

    fig,axs = plt.subplots(1,3,figsize=(8,2.5), layout="constrained")

    ## plot data
    for ax, PLOT_VAR in zip(axs, ["T_34", "T_m", "T_3"]):
        ax.plot(dSST_w.year, dSST_w[PLOT_VAR].mean("sample"), c="r")
        ax.plot(dSST_w.year, -dSST_c[PLOT_VAR].mean("sample"), c="b")

    ## formatting
    # ax.axhline(0, ls="--", c='gray', lw=1)
    src.utils.add_vticks(axs, xticks=[2010, 2050], xlines=[2010, 2050])
    # src.utils.set_lims(axs)
    # for ax in axs[1:]:
    #     ax.set_yticks([])
    plt.show()

In [ ]:
## specify plot var
PLOT_VAR = "T_3"

for s in [spags_warm, -spags_cold]:

    fig,ax = plt.subplots(figsize=(3,2.5))
    ax.plot(s.lag, s.isel(year=0)[PLOT_VAR].mean("sample"))
    ax.plot(s.lag, s.sel(year=2010)[PLOT_VAR].mean("sample"))
    ax.plot(s.lag, s.sel(year=2050)[PLOT_VAR].mean("sample"))
    src.utils.add_vticks([ax], xticks=[-6],xlines=[-6, 0])
    plt.show()
# plt.plot(spags_warm.lag, -spags_cold.isel(year=0)["T_3"].mean("sample"))